In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# Check if the faker library is already installed
try:
    from faker import Faker
except ImportError:
    !pip install faker
    from faker import Faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 20.2 MB/s eta 0:00:00


In [ ]:
# Initialize Faker with a seed for reproducible data
fake = Faker()
Faker.seed(42)
np.random.seed(42)

In [ ]:
from pathlib import Path

# Defining the data path to put the data in
data_path = str(Path.cwd() / "drive/MyDrive/Colab Notebooks/Projects/Ecommerce Data Synthesized/Data/")

### Generating users

Here, I created a realistic mock user dataset for analysis, prototyping, and testing data pipelines without using sensitive real-world personal information (PII).

I used the Faker library to generate realistic user demographics (names, emails, birthdates) and account creation timestamps over the past 3 years. And I also integrated `NumPy`'s weighted random choice to simulate realistic traffic source distributions  (e.g., Organic Search driving higher volume than Direct traffic).

In [ ]:
NUM_USERS = 25000
channels = ['Organic Search', 'Paid Search', 'Social Media', 'Email', 'Direct']
sex = ['male', 'female', 'non-binary']

users = []

for i in range(1, NUM_USERS + 1):
    # Create a random user profile that includes all general information
    user_profile = fake.simple_profile()
    created_at = fake.date_time_between(start_date="-3y", end_date="now")
    traffic_source = np.random.choice(channels, p=[0.3, 0.25, 0.2, 0.15, 0.1])

    users.append({
        'user_id': i,
        'created_at': created_at,
        'user_name':  user_profile['name'],
        'email': user_profile['mail'],
        'gender': user_profile['sex'],
        'birth_date': user_profile['birthdate'],
        'traffic_source': traffic_source
    })

In [ ]:
df_users = pd.DataFrame(users)
df_users.sample(5)

,user_id,created_at,user_name,email,gender,birth_date,traffic_source
21842,21843,2025-11-20 09:50:24.607839,Angela Jackson,michael19@hotmail.com,F,1933-03-12,Paid Search
5150,5151,2024-03-13 04:35:23.948704,Jillian Tran,dixontonya@yahoo.com,F,2025-04-19,Paid Search
16381,16382,2026-07-10 00:57:31.589394,Michael Simpson,floydsharon@hotmail.com,M,2016-06-07,Social Media
7282,7283,2025-12-12 00:50:02.929485,Lance Reed,nicholasdavidson@hotmail.com,M,1982-12-12,Organic Search
11714,11715,2024-12-02 03:19:46.417047,Amanda Sandoval,ucasey@gmail.com,F,1999-06-21,Organic Search


In [ ]:
# Save the dataset
df_users.to_csv(data_path + "/users.csv", index=False)

### Generating orders

Then, I simulated a realistic transactional e-commerce dataset tied to existing user profiles, allowing for order history analysis, sales revenue tracking, and order
status lifecycle modeling.

The code iterates through each user's assigned order count and constructs individual order records. It also ensures logical data consistency by forcing order timestamps to occur AFTER account creation dates (`user_created`). Then it simulates realistic purchase variability using random item counts, item-level pricing, tiered shipping fees, weighted discount probabilities, and realistic order status distributions.

In [ ]:
"""
USER ORDER FREQUENCY ALLOCATION
-------------------------------
The code establishes an order frequency baseline for each user to bridge user profiles  with transactional data, ensuring the overall user base exhibits a realistic customer purchasing distribution.

It extracts user identifiers and account creation dates from the user dataset,
then applies a weighted probability distribution using NumPy to assign total
order counts () Profile: 80% never buy, 14% buy once, 4% buy twice, 2% are repeat/power buyers)
"""
order_count_df = df_users[["user_id", "created_at"]].copy()
order_count_df["total_orders"] = np.random.choice([0, 1, 2, 3, 4, 5], size=len(order_count_df), p=[0.8, 0.14, 0.04, 0.012, 0.005, 0.003])

In [ ]:
# Shuffle the rows so orders aren't assigned in the same sequence as user IDs,
# preventing unrealistic pairings like user 1 always getting the first order, user 2 the second, etc.
order_count_df_shuffle = order_count_df.sample(frac=1, replace=False)

In [ ]:
order_count_df_shuffle.head(5)

,user_id,created_at,total_orders
13570,13571,2026-01-28 04:38:05.734110,0
17219,17220,2025-03-19 17:52:17.931569,0
4575,4576,2025-08-09 17:17:00.257807,0
20364,20365,2025-01-26 12:59:18.657100,0
19125,19126,2025-02-27 21:37:08.715815,0


In [ ]:
# Check the first user
for _, each in order_count_df_shuffle.iterrows():
    print(each)
    break

user_id                              13571
created_at      2026-01-28 04:38:05.734110
total_orders                             0
Name: 13570, dtype: object


In [ ]:
print("Repated Orders: ", sum(order_count_df_shuffle["total_orders"] > 1))
print("Non-Repeated Orders:", sum(order_count_df_shuffle["total_orders"] == 1))
print("No Order: ", sum(order_count_df_shuffle["total_orders"] == 0))
print("Total Orders: ", sum(order_count_df_shuffle["total_orders"]))

Repated Orders:  1427
Non-Repeated Orders: 3530
No Order:  20043
Total Orders:  7125


In [ ]:
# Genreate orders
statuses = ['completed', 'shipped', 'cancelled', 'returned']

orders = []

for _, each in order_count_df_shuffle.iterrows():
    # Get the user informatoin
    user_id = each["user_id"]
    user_created = each["created_at"]
    total_orders = each["total_orders"]

    # Generate number of items for customers who only made a purchase
    for i in range(1, total_orders + 1):
        # Order date after user_created date
        order_timestamp = fake.date_time_between(start_date=pd.to_datetime(user_created), end_date="now")

        item_count = random.randint(1, 5)

        # Since products can vary in prices, a random price is generated for each item before summing them up for the total amount
        total_amount = round(sum([random.uniform(15.00, 350.00) for each in range(1, item_count + 1)]), 2)

        shipping_fee = random.choice([0.00, 4.99, 9.99])
        discount = round(total_amount * random.choice([0, 0, 0, 0.1, 0.15, 0.2]), 2)

        # The status of the final stage of the order
        status = np.random.choice(statuses, p=[0.70, 0.15, 0.10, 0.05])

        orders.append({
            "user_id": user_id,
            "order_timestamp": order_timestamp,
            "total_amount": total_amount,
            "item_count": item_count,
            "shipping_fee": shipping_fee,
            "discount": discount,
            "status": status
        })

In [ ]:
df_orders = pd.DataFrame(orders)
df_orders.head(5)

,user_id,order_timestamp,total_amount,item_count,shipping_fee,discount,status
0,3991,2026-05-09 09:21:42.521066,402.42,3,9.99,60.36,shipped
1,527,2026-03-10 23:56:58.151002,305.73,1,4.99,0.00,completed
2,527,2026-05-02 00:45:04.697076,832.17,5,9.99,0.00,completed
3,527,2026-03-13 18:55:17.162656,533.45,3,4.99,0.00,completed
4,527,2026-07-30 16:09:55.166205,522.01,4,9.99,0.00,returned


The problem with the dataframe above is that it is looped through by the `user_id`. To micmic the real world, we need to create `order_id` based on the chronological timestamp.

In [ ]:
# Sort all orders chronologically by timestamp, generate a clean sequential index, and overwrite the dataframe
df_orders = df_orders.sort_values("order_timestamp").reset_index(drop=True)

In [ ]:
# Creat the order_id column
df_orders["order_id"] = range(1, len(df_orders) + 1)
df_orders.head()

,user_id,order_timestamp,total_amount,item_count,shipping_fee,discount,status,order_id
0,3131,2023-08-25 13:41:24.350941,1015.85,5,9.99,0.0,completed,1
1,7175,2023-08-31 17:15:00.849024,411.87,2,9.99,0.0,completed,2
2,17892,2023-09-08 11:54:27.860508,346.87,2,4.99,0.0,completed,3
3,18334,2023-09-10 20:22:42.379541,169.27,2,9.99,0.0,completed,4
4,3478,2023-09-12 14:56:46.803249,697.05,4,0.00,69.7,completed,5


In [ ]:
# Move the last column to the first
cols = df_orders.columns.tolist()
cols = cols[-1:] + cols[:-1]
df_orders = df_orders[cols]
df_orders.head()

,order_id,user_id,order_timestamp,total_amount,item_count,shipping_fee,discount,status
0,1,3131,2023-08-25 13:41:24.350941,1015.85,5,9.99,0.0,completed
1,2,7175,2023-08-31 17:15:00.849024,411.87,2,9.99,0.0,completed
2,3,17892,2023-09-08 11:54:27.860508,346.87,2,4.99,0.0,completed
3,4,18334,2023-09-10 20:22:42.379541,169.27,2,9.99,0.0,completed
4,5,3478,2023-09-12 14:56:46.803249,697.05,4,0.00,69.7,completed


In [ ]:
df_orders.shape

(7125, 8)

In [ ]:
df_orders.to_csv(data_path + "/orders.csv", index=False)

### Generating web logs

Finally, I generated clickstream and behavioral event logs to simulate user interactions across the conversion funnel (e.g., page views, add-to-cart, checkout steps), enabling session-level user journey, conversion rate, and drop-off analysis.

The code iterates through purchasing users to generate multiple browsing sessions per user. It also leverages lookups for creation dates and order counts to align session
timestamps logically, and then distinguishes between converting sessions (which complete  all 5 funnel steps) and non-converting sessions (which drop off early based on
a weighted probability distribution), tracking step-by-step timestamps, device types, and browsers per session.

**Methodology Note & Known Limitations:**
- Scope: Event logs are currently restricted to registered users in the dataset.
- Anonymous Users Excluced: anonymous users generate no web traffic in this dataset.
- Trade-off: This ensures strict parity between transactional order totals and completed conversion events, but underestimates top-of-funnel drop-off rates anonymous sessions.

In [ ]:
df_orders[df_orders["user_id"]==3131]

,order_id,user_id,order_timestamp,total_amount,item_count,shipping_fee,discount,status
0,1,3131,2023-08-25 13:41:24.350941,1015.85,5,9.99,0.0,completed
178,179,3131,2024-03-30 20:13:00.525548,462.89,2,0.00,0.0,cancelled


In [ ]:
df_orders[df_orders["user_id"]==2856]

,order_id,user_id,order_timestamp,total_amount,item_count,shipping_fee,discount,status


In [ ]:
event_types = ['page_view', 'add_to_cart', 'checkout_start', 'payment_info', 'purchase_complete']
devices = ['Mobile', 'Desktop', 'Tablet']
browsers = ['Chrome', 'Safari', 'Firefox', 'Edge']

events = []

# user_id: created_at dictionary for efficient searching
user_created_dict = df_users.set_index('user_id')['created_at'].to_dict()

# Get actual list of order_ids per user so we can map them to converted sessions
user_orders_dict = df_orders.groupby("user_id")["order_id"].apply(list).to_dict()

session_id = 1

for user_id in df_users["user_id"].unique():
    user_created = user_created_dict[user_id]
    user_orders = user_orders_dict.get(user_id, [])
    order_count = len(user_orders)

    extra_visits = np.random.choice([1, 2, 3, 4, 6, 8], p=[0.4, 0.3, 0.15, 0.08, 0.05, 0.02])
    visit_site_count = order_count + extra_visits

    # Generate all session start times first and sort them chronologically for this user
    session_starts = [
        fake.date_time_between(start_date=pd.to_datetime(user_created), end_date="now")
        for _ in range(visit_site_count)
    ]
    session_starts.sort()

    for i, session_start in enumerate(session_starts, start=1):
        device_type = np.random.choice(devices, p=[0.6, 0.3, 0.1])
        browser = np.random.choice(browsers, p=[0.7, 0.2, 0.05, 0.05])

        # Converting sessions get assigned a real order_id on the final step
        if i <= order_count:
            depth = len(event_types)
            current_order_id = user_orders[i-1]
        else:
            depth = random.choices([1, 2, 3, 4], weights=[0.4, 0.25, 0.15, 0.1])[0]
            current_order_id = None

        step_time = session_start

        for step in range(depth):
            step_time += timedelta(seconds=random.randint(10, 120))

            # Order ID is only attached to the 'purchase_complete' event
            is_purchase_step = (event_types[step] == 'purchase_complete')

            events.append({
                'session_id': session_id,
                'user_id': user_id,
                'order_id': current_order_id if is_purchase_step else None,
                'event_type': event_types[step],
                'event_timestamp': step_time,
                'device_type': device_type,
                'browser': browser
            })

        session_id += 1

In [ ]:
# Convert to DataFrame, sort globally by time, and assign strictly sequential event_id
df_web_logs = pd.DataFrame(events)
df_web_logs = df_web_logs.sort_values(by="event_timestamp").reset_index(drop=True)

In [ ]:
df_web_logs["event_id"] = df_web_logs.index + 1

# Reorder columns to put event_id first
cols = ['event_id', 'session_id', 'user_id', 'order_id', 'event_type', 'event_timestamp', 'device_type', 'browser']
df_web_logs = df_web_logs[cols]

In [ ]:
df_web_logs.sample(5)

,event_id,session_id,user_id,event_type,event_timestamp,device_type,browser
4045,4046,1766,714,add_to_cart,2026-04-05 18:08:17.206087,Tablet,Firefox
52419,52420,22925,9135,checkout_start,2026-03-31 21:54:18.652565,Tablet,Chrome
80586,80587,35262,13965,add_to_cart,2026-03-15 16:28:56.944520,Mobile,Edge
47600,47601,20791,8284,page_view,2026-03-21 17:13:32.188528,Mobile,Chrome
23148,23149,10089,4029,checkout_start,2025-11-30 08:09:30.384489,Tablet,Safari


In [ ]:
df_web_logs.shape

(144178, 7)

In [ ]:
df_web_logs.to_csv(data_path + "/web_logs.csv", index=False, na_rep="NULL")

In [ ]:
df_web_logs[df_web_logs["user_id"]==3131]["event_type"].value_counts()

,count
event_type,
page_view,8
add_to_cart,5
checkout_start,3
payment_info,2
purchase_complete,2


In [ ]:
df_orders[df_orders["user_id"]==3131]

,order_id,user_id,order_timestamp,total_amount,item_count,shipping_fee,discount,status
0,1,3131,2023-08-25 13:41:24.350941,1015.85,5,9.99,0.0,completed
178,179,3131,2024-03-30 20:13:00.525548,462.89,2,0.00,0.0,cancelled


In [ ]:
df_users = pd.read_csv(data_path + "/users.csv")
df_orders = pd.read_csv(data_path + "/orders.csv")
df_web_logs = pd.read_csv(data_path + "/web_logs.csv")

In [ ]:
df_users.shape[0], df_orders.shape[0], df_web_logs.shape[0]

(25000, 7125, 143919)